Aquí tienes la traducción técnica adaptada al español, las correcciones y la actualización para el estado del arte actual (2026), junto con los scripts modernos de Python para visualizar la Ley de los Grandes Números y las fronteras de decisión de los clasificadores.

---

## 1. Traducción, Corrección y Adaptación del Texto

### Clasificadores por Votación

Supón que has entrenado unos cuantos clasificadores y cada uno alcanza aproximadamente un 80% de precisión (*accuracy*). Podrías tener un clasificador de regresión logística, un clasificador SVM, uno de bosques aleatorios, uno de $k$-vecinos más cercanos y quizás algunos más.

Una forma muy sencilla de crear un clasificador aún mejor consiste en agregar las predicciones de cada uno: la clase que obtenga la mayoría de los votos será la predicción del ensamble. Este clasificador por mayoría de votos se denomina **clasificador por votación estricta (hard voting classifier)**.

Sorprendentemente, este clasificador por votación suele alcanzar una precisión mayor que la del mejor clasificador individual del ensamble. De hecho, incluso si cada clasificador es un **aprendiz débil (weak learner)** —lo que significa que su rendimiento es solo ligeramente mejor que el azar—, el ensamble puede convertirse en un **aprendiz fuerte (strong learner)**, siempre que haya un número suficiente de aprendices débiles en el ensamble y estos sean lo suficientemente diversos (es decir, si se enfocan en diferentes aspectos de los datos y cometen distintos tipos de errores).

¿Cómo es esto posible? La siguiente analogía ayuda a esclarecer este misterio. Supón que tienes una moneda ligeramente cargada que tiene un 51% de probabilidad de salir cara y un 49% de salir cruz. Si la lanzas 1,000 veces, por lo general obtendrás más o menos 510 caras y 490 cruces, y por ende, una mayoría de caras. Si haces las matemáticas, verás que la probabilidad de obtener una mayoría de caras después de 1,000 lanzamientos es cercana al 75%. Cuanto más lances la moneda, mayor será esa probabilidad (por ejemplo, con 10,000 lanzamientos, la probabilidad supera el 97%).

Esto se debe a la **ley de los grandes números**: a medida que continúas lanzando la moneda, la proporción de caras se acerca cada vez más a su probabilidad real (51%). La Figura 6-3 muestra 10 series de lanzamientos de monedas cargadas. Puedes ver que, a medida que aumenta el número de lanzamientos, la proporción de caras se aproxima al 51%, hasta que finalmente las 10 series se estabilizan consistentemente por encima del 50%.

Del mismo modo, supón que construyes un ensamble que contiene 1,000 clasificadores que individualmente son correctos solo el 51% de las veces (apenas mejor que el azar). Si predices mediante el voto de la mayoría, ¡podrías aspirar a una precisión de hasta el 75%! Sin embargo, esto solo es estrictamente cierto si todos los clasificadores son **perfectamente independientes** y cometen errores no correlacionados. En la práctica esto claramente no ocurre porque se entrenan con los mismos datos. Es probable que cometan los mismos tipos de errores, por lo que habrá muchos votos mayoritarios dirigidos a la clase incorrecta, reduciendo la precisión del ensamble.

> Los métodos de ensamble funcionan mejor cuando los predictores son lo más independientes posible entre sí. Una forma de obtener clasificadores diversos es entrenarlos utilizando algoritmos muy diferentes. Esto aumenta la probabilidad de que cometan tipos de errores muy distintos, mejorando la precisión del ensamble. También puedes jugar con los hiperparámetros de los modelos o entrenarlos en diferentes subconjuntos de datos.

Scikit-Learn proporciona una clase llamada `VotingClassifier` que es muy fácil de usar: solo debes pasarle una lista de pares nombre/predictor y utilizarla como un clasificador normal. Probémoslo en el conjunto de datos *moons*. Cargaremos y dividiremos el dataset en entrenamiento y prueba, y luego crearemos un clasificador por votación compuesto por tres algoritmos diversos:

```python
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(random_state=42))
    ]
)
voting_clf.fit(X_train, y_train)

```

Cuando ajustas un `VotingClassifier`, este clona cada estimador y entrena los clones. Los estimadores originales están disponibles a través del atributo `estimators`, mientras que los clones entrenados están en `estimators_`. Si prefieres un diccionario en lugar de una lista, puedes usar `named_estimators_`. Veamos la precisión de cada clasificador entrenado en el conjunto de prueba:

```python
for name, clf in voting_clf.named_estimators_.items():
    print(name, "=", clf.score(X_test, y_test))

```

*Salida:*

```text
lr = 0.864
rf = 0.896
svc = 0.896

```

Cuando llamas al método `predict()` del clasificador por votación, este realiza una **votación estricta (hard voting)**. Por ejemplo, predice la clase 1 para la primera instancia del conjunto de prueba porque dos de los tres clasificadores eligen esa clase:

```python
print(voting_clf.predict(X_test[:1]))
# array([1])

print([clf.predict(X_test[:1]) for clf in voting_clf.estimators_])
# [array([1]), array([1]), array([0])]

```

Evaluemos el rendimiento global del clasificador por votación en el conjunto de prueba:

```python
voting_clf.score(X_test, y_test)
# 0.912

```

¡Ahí lo tienes! El clasificador por votación supera a todos los clasificadores individuales.

Si todos los modelos son capaces de estimar probabilidades de clase (es decir, si cuentan con el método `predict_proba()`), puedes indicarle a Scikit-Learn que prediga la clase con la probabilidad promedio más alta. Esto se conoce como **votación atenuada o blanda (soft voting)**. Suele ofrecer un rendimiento superior a la votación estricta porque otorga mayor peso a los votos en los que los modelos muestran una alta confianza.

Todo lo que necesitas hacer es configurar el hiperparámetro `voting="soft"` y asegurarte de que todos los modelos estimen probabilidades. La clase `SVC` no lo hace por defecto, por lo que debes configurar su parámetro `probability=True` (esto obligará a la clase `SVC` a usar validación cruzada interna para estimar probabilidades, lo que ralentiza el entrenamiento). Intentémoslo:

```python
# Modificando hiperparámetros dinámicamente y reentrenando
voting_clf.voting = "soft"
voting_clf.named_estimators["svc"].probability = True
voting_clf.fit(X_train, y_train)
print(voting_clf.score(X_test, y_test))
# 0.92

```

¡Alcanzamos un 92% de precisión simplemente cambiando a votación atenuada!

---

## 2. Actualizaciones Técnicas y del Estado del Arte (2026)

* **Calibración de Probabilidades con Modelos Modernos:** Géron menciona al final que las probabilidades deben estar bien calibradas usando `CalibratedClassifierCV`. En 2026, esto es un estándar mandatorio si se incluyen algoritmos como **XGBoost, LightGBM o redes neuronales profundas** en el `VotingClassifier`. Estos modelos sufren de sobreconfianza estructural (tienden a predecir probabilidades extremas como `0.99` o `0.01` aunque estén equivocados), lo que rompe por completo la matemática del *soft voting* si no se calibran previamente mediante regresión isotónica o el método de Platt.
* **Buenas prácticas de mutabilidad:** Modificar los hiperparámetros directamente sobre el objeto ya construido mediante asignaciones directas como `voting_clf.voting = "soft"` puede generar inconsistencias en los clones internos si se usan flujos de trabajo avanzados. La forma idiomática y moderna de Scikit-Learn para cambiar hiperparámetros antes de un reentrenamiento es mediante el método `.set_params()`:
```python
voting_clf.set_params(voting="soft", svc__probability=True)

```



---

## 3. Scripts de Python para Ejemplificar y Visualizar

### Script 1: Simulación Matemática de la Ley de los Grandes Números (Figura 6-3)

Este script recrea de forma exacta el experimento estadístico del lanzamiento de monedas sesgadas ($51\%$) para demostrar visualmente cómo el volumen de intentos estabiliza la probabilidad.

```python
import numpy as np
import matplotlib.pyplot as plt

# Configuración de la simulación
rng = np.random.default_rng(seed=42)
prob_caras = 0.51
n_lanzamientos = 10000
n_series = 10

# Simular 10 series de 10,000 lanzamientos (1 = Cara, 0 = Cruz)
lanzamientos = rng.binomial(1, prob_caras, size=(n_series, n_lanzamientos))
# Calcular la proporción acumulada de caras paso a paso
proporciones_acumuladas = np.cumsum(lanzamientos, axis=1) / np.arange(1, n_lanzamientos + 1)

# Graficar
plt.figure(figsize=(12, 6))
for i in range(n_series):
    plt.plot(proporciones_acumuladas[i], alpha=0.8)

plt.axhline(y=0.51, color="red", linestyle="--", linewidth=2, label="Probabilidad Real (51%)")
plt.axhline(y=0.50, color="black", linestyle="-", linewidth=1)
plt.title("Figura 6-3: La Ley de los Grandes Números usando Monedas Sesgadas", fontsize=13, fontweight='bold')
plt.xlabel("Número de lanzamientos de moneda", fontsize=11)
plt.ylabel("Proporción de caras", fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(loc="lower right")
plt.axis([0, n_lanzamientos, 0.42, 0.58])
plt.show()

```

---

### Script 2: Fronteras de Decisión en Scikit-Learn: Hard vs Soft Voting

Este script genera el dataset de las dos lunas, calibra los modelos utilizando la sintaxis `.set_params()` y grafica comparativamente cómo se comportan las fronteras de decisión bajo votación estricta frente a la votación probabilística.

```python
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC

# 1. Preparar datos
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# 2. Definir el clasificador por votación (Hard)
voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(probability=True, random_state=42)) # probability=True requerido para soft posterior
    ],
    voting='hard'
)
voting_clf.fit(X_train, y_train)
score_hard = voting_clf.score(X_test, y_test)

# 3. Clonar y cambiar parámetros usando la nomenclatura idiomática .set_params() para Soft Voting
voting_clf_soft = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(probability=True, random_state=42))
    ],
    voting='soft'
)
voting_clf_soft.fit(X_train, y_train)
score_soft = voting_clf_soft.score(X_test, y_test)

# 4. Configurar mallas de visualización
x1s = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 500)
x2s = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 500)
x1, x2 = np.meshgrid(x1s, x2s)
X_mesh = np.c_[x1.ravel(), x2.ravel()]

# Predicciones de las mallas
y_pred_hard = voting_clf.predict(X_mesh).reshape(x1.shape)
y_pred_soft = voting_clf_soft.predict(X_mesh).reshape(x1.shape)

# Graficar comparativa de fronteras
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
cmap_back = ListedColormap(['#ffaaaa', '#aaaaff'])
cmap_pts = ListedColormap(['red', 'blue'])

# Gráfica de Hard Voting
axes[0].contourf(x1, x2, y_pred_hard, alpha=0.3, cmap=cmap_back)
axes[0].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cmap_pts, edgecolor='k', alpha=0.7)
axes[0].set_title(f"Hard Voting Classifier\nPrecisión en Test: {score_hard*100:.1f}%", fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Gráfica de Soft Voting
axes[1].contourf(x1, x2, y_pred_soft, alpha=0.3, cmap=cmap_back)
axes[1].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cmap_pts, edgecolor='k', alpha=0.7)
axes[1].set_title(f"Soft Voting Classifier (Probabilidades promedio)\nPrecisión en Test: {score_soft*100:.1f}%", fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle("Comparativa de Estrategias de Votación en Ensambles", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

```